# Data Exploration & Pipeline Verification

**Proiect:** Sistem de Numărare și Estimare a Densității de Mulțimi  
**Echipă:** Constantin Florin-George & Jugaru Cosmin-Alexandru  
**Materia:** Analiza Conținutului Imaginilor  

Acest notebook permite verificarea mediului de lucru și a loader-ului PyTorch pentru setul de date ShanghaiTech.

In [ ]:
# 1. Verificare mediu (GPU si biblioteci)
import torch, sys
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Director de lucru
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
%cd {PROJECT_ROOT}
print('Working directory:', Path.cwd())

In [ ]:
# 3. Verificare sumar manifesturi
import json
with open('data/processed/manifests/audit_summary.json', 'r', encoding='utf-8') as f:
    summary = json.load(f)
for part, data in summary.get('parts', {}).items():
    print(f"{part}: {data['valid_samples']} imagini valide, {data['crowd_stats']['total_people_annotations']} persoane adnotate")

In [ ]:
# 4. Verificare PyTorch Dataset & DataLoader
from torch.utils.data import DataLoader
from src.data.pytorch_dataset import ShanghaiTechCrowdDataset

train_ds = ShanghaiTechCrowdDataset(
    manifest_csv='data/processed/manifests/part_A_train.csv',
    project_root='.',
    augment=True,
    downsample_factor=8,
    as_torch=True,
)
val_ds = ShanghaiTechCrowdDataset(
    manifest_csv='data/processed/manifests/part_A_val.csv',
    project_root='.',
    augment=False,
    downsample_factor=8,
    as_torch=True,
)

sample = val_ds[0]
print(f'Train samples: {len(train_ds)} | Val samples: {len(val_ds)}')
print('Image shape:', tuple(sample['image'].shape))
print('Density shape (stride 8):', tuple(sample['density'].shape))
print(f"GT count: {sample['count'].item():.2f} | Density integral: {sample['density'].sum().item():.2f}")

val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)
batch = next(iter(val_loader))
print('Loader test OK:', batch['sample_id'])